# Esri services on a geeViz map

geeViz draws ArcGIS image, map and feature services beside Earth Engine layers.
`Map.addLayer` takes a service URL the same way it takes an `ee.Image`, and
`Map.addTimeLapse` plays a time-enabled image service year by year. Nothing is
fetched in Python: geoView, geeViz's map viewer, asks the services itself.

You rarely need to know a service's URL. georest, which geeViz installs, searches
ArcGIS portals and the Forest Service Enterprise Data Warehouse (EDW) for you.
Its own examples at [georest.io](https://georest.io/examples.html) cover searching,
service metadata and queries in depth; this notebook is about putting what you
find on a geeViz map.

In [1]:
import geeViz.geeView as gv
from georest.esri import portal
from georest.usfs import edw

ee = gv.ee
Map = gv.Map

[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:8889/ee-api (attached to existing detached proxy, tenant_header=X-geeViz-Creds)


geeViz: Earth Engine ready (project='geeviz-geo-agent', source=eeauth-proxy)


## Find the services

Search IIPP, the federal imagery portal, for LCMS (the Landscape Change Monitoring
System), and pick the annual change service for the lower 48 states.

In [2]:
found = portal.searchPortal("LCMS annual change", limit=20)
for r in found:
    print(r["type"], "|", r["title"])

lcms_change = next(r["url"] for r in found if r["type"] == "Image Service" and r["title"].endswith("AnnualChange_CONUS"))
print()
print("Using", lcms_change)

Image Service | USFS_EDW_LCMS_AnnualLandcover_CONUS
Image Service | USFS_EDW_LCMS_AnnualChange_HI
Image Service | USFS_EDW_LCMS_AnnualQABits_AK
Image Service | USFS_EDW_LCMS_AnnualLandcover_AK
Image Service | USFS_EDW_LCMS_AnnualChange_CONUS
Image Service | USFS_EDW_LCMS_AnnualLanduse_CONUS
Image Service | USFS_EDW_LCMS_AnnualLanduse_PRUSVI
Image Service | USFS_EDW_LCMS_AnnualLanduse_AK
Image Service | USFS_EDW_LCMS_AnnualLandcover_HI
Image Service | USFS_EDW_LCMS_AnnualQABits_HI
Image Service | USFS_EDW_LCMS_AnnualQABits_PRUSVI
Image Service | USFS_EDW_LCMS_AnnualLandcover_PRUSVI
WMS | USFS_EDW_LCMS_AnnualChange_CONUS
Image Service | USFS_EDW_LCMS_YearHighestProbabilityGain_PRUSVI
Image Service | USFS_EDW_LCMS_AnnualQABits_CONUS
Image Service | USFS_EDW_LCMS_MostRecentYearGain_CONUS
Image Service | USFS_EDW_LCMS_YearHighestProbabilityGain_CONUS
Image Service | USFS_EDW_LCMS_MostRecentYearGain_PRUSVI
Image Service | USFS_EDW_LCMS_AnnualChange_PRUSVI
Image Service | USFS_EDW_LCMS_YearHi

Search EDW for MTBS (Monitoring Trends in Burn Severity), then find the burned
area boundaries for all years among the service's layers.

In [3]:
mtbs = edw.search_edw_services("MTBS")
for s in mtbs:
    print(s["name"], "|", s["url"])

layers = portal.getServiceMetadata(mtbs[0]["url"])["layers"]
boundaries = next(l for l in layers if l["name"] == "Burned Area Boundaries (All Years)")
mtbs_fires = "{}/{}".format(mtbs[0]["url"], boundaries["id"])
print()
print("Using", mtbs_fires)

EDW_MTBS_01 | https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_MTBS_01/MapServer



Using https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_MTBS_01/MapServer/63


## Put them on the map with Earth Engine

The Cameron Peak fire burned west of Fort Collins, Colorado, in 2020. The map has
the LCMS change service as a time lapse, the MTBS perimeters, and MODIS burned
area from Earth Engine for the same year. Turn on the Query tool and click a
perimeter for its attributes, or use Chart area on the time lapse for change over
time, as a line chart and a Sankey.

In [4]:
Map.clearMap()
Map.setSkin("geoview")

Map.addTimeLapse(lcms_change, {"canAreaChart": True, "areaChartParams": {"line": True, "sankey": True}}, "LCMS annual change")

burned = ee.ImageCollection("MODIS/061/MCD64A1").filterDate("2020-01-01", "2021-01-01").select("BurnDate").max().selfMask()
Map.addLayer(burned, {"min": 150, "max": 300, "palette": ["ffd400", "ff7b00", "b30000"]}, "MODIS burned area 2020 (day of year)")

Map.addLayer(mtbs_fires, {"where": "year >= 2015", "canAreaChart": True, "chartField": "fire_type"}, "MTBS fires since 2015")

Map.setCenter(-105.65, 40.62, 10)
Map.turnOnInspector()
Map.view()

Adding layer: LCMS annual change
Adding layer: MODIS burned area 2020 (day of year)
Adding layer: MTBS fires since 2015
Starting webmap


Using eeCreds proxy at http://127.0.0.1:8889/ee-api (creds=ee-persistent, mode=detached via default)
geeView URL: http://127.0.0.1:8889/geoview/geoview.html?v=1795666368800&doc=../geeView/src/gee/gee-run/map-a614eb977097536f-00274767837b.json


## Secured services

For a service that needs an ArcGIS token, pass it to `Map.addEsriService`, which
keeps the token in the local server instead of the page. `Map.addLayer` has no
token argument.

```python
Map.addEsriService("https://your-portal.gov/server/rest/services/Private/MapServer", token=my_token)
```